In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable


In [0]:
%run /Workspace/consolidated-pipeline/1st_setup/Utilities

In [0]:
dbutils.widgets.text("Catalog","fmcg","catalog")
dbutils.widgets.text("Data_Source","products","data_source")

In [0]:
catalog= dbutils.widgets.get("Catalog")
data_source = dbutils.widgets.get("Data_Source")
print(catalog,data_source)
base_path = f"s3://amzn-s3-sportsbar/{data_source}/*.csv"
print(base_path)

##Bronze Processing

In [0]:
df= (
    spark.read.format("csv")
    .option("header",True)
    .option("inferSchema",True)
    .option("mode","PERMISSIVE")
    .option("multiline",True)
    .load(base_path)
    .withColumn("readTimeStamp",F.current_timestamp())
    .select("*","_metadata.file_name", "_metadata.file_size")
    )

In [0]:
display(df)

In [0]:
df.printSchema()

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed",True)\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

##Silver Processing


In [0]:
df_bronze = spark.sql(f"Select * From {catalog}.{bronze_schema}.{data_source};")
display(df_bronze.limit(5))

In [0]:
df_bronze.groupBy(df_bronze.columns).count().filter("count>1").show()

Dropping Duplicates


In [0]:
print("Total rows before droppping duplicates", df_bronze.count())
df_silver = df_bronze.drop_duplicates(["product_id"])
print("Total rows after droppping duplicates", df_silver.count())

In [0]:
df_silver.filter(F.col("product_id").isNull()).show()

Title case fix, 
energy bars -> Energy Bars
protien bar -> Protein Bar etc


In [0]:
df_silver.select("category").distinct().show()

In [0]:
df_silver= df_silver.withColumn(
    "category",
    F.when(F.col("category").isNull(), None)
    .otherwise(F.initcap("category"))
)
df_silver.select("category").distinct().show()

In [0]:
df_silver = (df_silver.withColumn(
    "product_name",
    F.regexp_replace(F.col("product_name"),"(?i)protien","Protein")
).withColumn(
    "category",
    F.regexp_replace(F.col("category"),"(?i)protien","Protein")
    )
)
display(df_silver)

Standradizing Customer Attribute to match parent company data model

In [0]:
df_silver=(
    df_silver.withColumn(
        "division",
        F.when(F.col("category")=="Energy Bars","Nutrition Bars")
        .when(F.col("category")=="Protein Bars","Nutrition Bars")
        .when(F.col("category")=="Granola & Cereals","Breakfast Foods")
        .when(F.col("category")=="Recovery Dairy","Dairy & Recovery")
        .when(F.col("category")=="Healthy Snacks","Healthy Snacks")
        .when(F.col("category")=="Electrolyte Mix","Hydration & Electrolytes")
        .otherwise("Other")

    )
)

In [0]:
df_silver = (df_silver.withColumn(
    "variant",
    F.regexp_extract(F.col("product_name"),r"\((.*?)\)", 1)
))

#### 3: Create new column: product_code  

#### Invalid product_ids are replaced with a fallback value to avoid losing fact records and ensure downstream joins remain consistent


In [0]:
df_silver = (
  df_silver.withColumn(
    "product_code",
    F.sha2(F.col("product_name").cast("string"),256)
  ).withColumn(
    "product_id",
    F.when(
      F.col("product_id").cast("string").rlike("^[0-9]+$"),
      F.col("product_id").cast("string")
    ).otherwise(F.lit(999999).cast("string"))
  ).withColumnRenamed("product_name","product")
)

In [0]:
display(df_silver)

In [0]:
df_silver.printSchema()

In [0]:
df_silver = df_silver.select("product_code", "division", "category", "product", "variant", "product_id", "readTimeStamp", "file_name", "file_size")

In [0]:
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed",True)\
    .option("mergeSchema",True)\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

# Gold Processing

In [0]:

df_silver_processed_data = spark.sql(f"Select * from {catalog}.{silver_schema}.{data_source}")

In [0]:
df_gold=df_silver_processed_data.select("product_code","product_id","division","category","product", "variant")
display(df_gold)

In [0]:
df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed",True)\
    .option("mergeSchema",True)\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")


Merge Data Fraom with Original Data Table

In [0]:
delta_table=DeltaTable.forName(spark,"fmcg.gold.dim_products")
df_child_product =spark.table("fmcg.gold.sb_dim_products")\
.select("product_code","product_id","division","category","product", "variant")

In [0]:
delta_table.alias("target").merge(
    source= df_child_product.alias("source"),
    condition = "target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).execute()